In [23]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
ds_api = user_secrets.get_secret("deepseek_api")

In [16]:
from huggingface_hub import HfApi

api = HfApi(token=secret_value_0)

files = api.list_repo_files(
    repo_id="stukenov/sozkz-corpus-clean-v3",
    repo_type="dataset"
)

shards = sorted(
    f for f in files
    if f.startswith("data/shard_") and f.endswith(".parquet")
)

# smq:
# s = 19, m = 13, q = 17
# 2 is the number I like
# 2^19 = 524288 -> [1] = 5
# 2^13 = 8192   -> [2] = 1
# 2^17 = 131072 -> [3] = 1
# therefore: seed = 511
random.seed(511)
sampled_shards = random.sample(shards, 30)

print(sampled_shards)

['data/shard_00055.parquet', 'data/shard_00234.parquet', 'data/shard_00062.parquet', 'data/shard_00073.parquet', 'data/shard_00124.parquet', 'data/shard_00249.parquet', 'data/shard_00108.parquet', 'data/shard_00141.parquet', 'data/shard_00070.parquet', 'data/shard_00066.parquet', 'data/shard_00014.parquet', 'data/shard_00157.parquet', 'data/shard_00041.parquet', 'data/shard_00269.parquet', 'data/shard_00038.parquet', 'data/shard_00170.parquet', 'data/shard_00208.parquet', 'data/shard_00167.parquet', 'data/shard_00105.parquet', 'data/shard_00130.parquet', 'data/shard_00025.parquet', 'data/shard_00090.parquet', 'data/shard_00009.parquet', 'data/shard_00182.parquet', 'data/shard_00227.parquet', 'data/shard_00047.parquet', 'data/shard_00198.parquet', 'data/shard_00053.parquet', 'data/shard_00189.parquet', 'data/shard_00158.parquet']


In [20]:
rows = []

for shard in sampled_shards:
    
    ds = datasets.load_dataset(
        "stukenov/sozkz-corpus-clean-v3",
        data_files={"train": [shard]},
        split="train",
        verification_mode="no_checks",
        token = secret_value_0
    )

    sample = random.sample(ds['text'], 100)
    rows.extend(sample)


# print(rows)
print(len(rows))

6000


In [22]:
count_chars = sum([len(row) for row in rows])
count_chars

17943549

In [24]:
system_prompt = """
```text id="58321"
# Task: Forensic Audit of a Dataset Sample for LLM Pretraining

You are analyzing a random sample of documents from a large text dataset intended for pretraining a large language model (LLM) on the Kazakh language.

Your task is to conduct a detailed forensic audit of the provided sample and identify actual data-quality problems.

## Constraints

- Do not write code.
- Do not propose a preprocessing pipeline.
- Do not suggest how to fix or clean the detected problems.
- Do not make claims about the entire dataset based solely on this sample.
- Do not invent statistics or problems that cannot be supported by the provided data.
- Clearly distinguish directly observed facts from assumptions or hypotheses.
- If there is insufficient evidence to confirm a particular issue, explicitly state that.

## What to Analyze

Check the sample for the following types of problems:

### 1. Unicode and Encoding Issues

- Replacement character `�`;
- Invisible characters;
- Control characters;
- Non-standard whitespace;
- Suspicious Unicode sequences;
- Unicode confusables;
- Other signs of text corruption.

### 2. Script Mixing

- Latin characters inside Cyrillic words;
- Cyrillic characters inside Latin words;
- Mixed Unicode scripts within words;
- Cases such as `бiр`, `үшiн`, `мемлекеттiк`;
- Other systematic substitutions of visually similar characters.

### 3. Structural Artifacts

- Literal `\n`, `\t`, and other escape sequences;
- Broken line breaks;
- Repeated characters;
- Truncated text;
- Corrupted document structure;
- Other structural artifacts.

### 4. Web Artifacts

- HTML tags;
- URLs;
- Tracking parameters;
- Advertisements;
- Navigation/menu text;
- Cookie notices;
- Headers/footers;
- Social media widgets;
- Other web boilerplate.

### 5. Repetition and Duplication

- Exact duplicates;
- Repeated paragraphs;
- Repeated sentences;
- Near-duplicates;
- Syndicated or reposted content;
- Template-based documents;
- Excessive repetition within individual documents.

### 6. Language Quality

- Documents that are not actually Kazakh;
- Russian, English, or other languages;
- Excessive code-switching;
- Suspicious machine-translated text;
- Unnatural or corrupted Kazakh;
- OCR-like errors.

### 7. Overall Text Integrity

- Meaningless or nonsensical text;
- Corrupted documents;
- Truncated documents;
- Documents consisting primarily of metadata;
- Extremely short or suspiciously long documents;
- Other anomalies that could negatively affect the quality of an LLM pretraining corpus.

### 8. Additional Problems

Independently identify any other systematic anomalies that may be relevant to the quality of the corpus for LLM pretraining.

Do not limit the analysis to the categories listed above.

## Analysis Format

For every detected problem, use the following structure:

### [Problem Name]

**Problem:**  
Briefly and precisely describe what was detected.

**Examples:**  
Provide concrete excerpts from the provided sample demonstrating the problem.

**Scale within the sample:**  
Estimate how widespread the problem appears to be within the provided data.

If an exact frequency cannot be determined, use a qualitative assessment:

- Isolated case;
- Rare;
- Occasional;
- Common;
- Widespread;
- Very widespread / pervasive.

Do not invent numerical values if they cannot be reliably determined from the provided data.

**Potential impact on pretraining:**  
Rate the potential impact as:

- Critical
- High
- Medium
- Low
- Negligible

Briefly explain the reasoning behind the rating.

**Status:**  
Use exactly one of the following:

- `OBSERVED` — the problem is directly confirmed by examples in the sample;
- `LIKELY SYSTEMATIC` — the problem appears repeatedly and looks systematic, but its prevalence across the entire dataset is unknown;
- `UNCERTAIN` — there are suspicious indicators, but there is insufficient evidence to confidently classify it as a real problem.

**Conclusion:**  
State whether this issue should be checked across the entire dataset.

## Final Report

After completing the detailed analysis, provide a concise summary.

### Overall Assessment

Describe the main problems found and the overall quality of the sample from the perspective of its suitability for LLM pretraining.

Do not provide a quality assessment of the entire dataset if it cannot be justified by the sample.

### Problems to Check Across the Entire Dataset

Provide a prioritized list of the most important detected problems:

1. ...
2. ...
3. ...
4. ...
5. ...

Rank them based on both:

- potential impact on pretraining quality;
- confidence that the problem genuinely exists.

### Important Limitation

All conclusions must refer specifically to the provided sample.

Finding a problem in the sample means only:

> "This problem is present in the analyzed sample."

It does **not** automatically mean:

> "This problem is present throughout the entire dataset."

Therefore, do not extrapolate findings to the entire corpus without sufficient evidence.
```



"""

In [31]:
rows[3:5]

['Тас храмы немесе бірінші (унитарлық) қауымдық шіркеу - екі Адамзаның жерленген орны.',
 'Басқа дүңгіршектен Джули деген дауыс шықты, бірақ мен оны да аштым да, әйелдің тамағына пышақ салып, киімін реттеп, менімен бірге жүруін сабырмен айттым.']

In [36]:
parts = [rows[i:i+600] for i in range(0, len(rows), 600)]
len(parts)

10

In [37]:
import os
from tqdm import tqdm
from openai import OpenAI

client = OpenAI(
    api_key=ds_api,
    base_url="https://api.deepseek.com")

answers = []

for part in tqdm(parts):
    prompt = "\n\n".join(part)

    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        stream=False,
        reasoning_effort="high",
        extra_body={"thinking": {"type": "enabled"}}
    )

    answer = response.choices[0].message.content
    answers.append(answer)

 50%|█████     | 5/10 [11:07<11:07, 133.47s/it]


BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 1048576 tokens. However, you requested 1061006 tokens (1061006 in the messages, 0 in the completion). Please reduce the length of the messages or completion.", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

In [39]:
len(answers)

5

In [41]:
import json

with open("llm_valuation.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, ensure_ascii=False, indent=2)